# Ranking Model Evaluation and Visualization

This notebook provides comprehensive evaluation tools for trained SiameseRanker models used in floor plan evacuation quality prediction.

## Contents
1. Setup & Model Loading
2. Pairwise Evaluation (Loss, Accuracy, AUC)
3. Surrogate vs Simulation Comparison
4. Per-Plan Ranking Evaluation
5. GradCAM Visualization
6. Summary & Export

## Required Files
- Trained model checkpoint (`best_model.pt` or `latest_model.pt`)
- `scenario_stats.json` (normalization statistics)
- Test data (`test_pairs.jsonl`, `simulation_results.jsonl`)

---
## 1. Setup & Model Loading

In [ ]:
# Standard library
import json
import sys
from pathlib import Path
from collections import defaultdict
from typing import Dict, List, Optional, Tuple

# Scientific computing
import numpy as np
import pandas as pd
from scipy.stats import kendalltau, spearmanr, pearsonr

# Deep learning
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, HTML

# Metrics
from sklearn.metrics import roc_auc_score, roc_curve

# Project imports - add project root to path
project_root = Path.cwd().parent.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from ml.ranking.config import RankingConfig
from ml.ranking.model import SiameseRanker
from ml.ranking.dataset import PairwiseDataset, SingleConfigDataset, compute_scenario_stats
from ml.ranking.evaluate import compute_ndcg
from ml.ranking.visualize import GradCAM
from ml.ranking.losses import RankNetLoss

# Configure matplotlib
plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print("Imports complete!")

In [ ]:
# === CONFIGURATION - EDIT THESE PATHS ===
CHECKPOINT_PATH = "checkpoints/ranking/best_model.pt"  # Path to model checkpoint (file or directory)
DATA_DIR = Path("combined_fast")                       # Data directory with pairs and simulation results
FLOOR_PLANS_DIR = DATA_DIR / "floor_plans"             # Floor plans directory

# Device selection
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

# Evaluation settings
EVAL_BATCH_SIZE = 256

In [ ]:
def load_model_and_config(checkpoint_path: str, device: torch.device) -> Tuple[SiameseRanker, RankingConfig, Dict]:
    """Load model, config, and checkpoint data."""
    checkpoint_path = Path(checkpoint_path)
    
    # Handle directory input (auto-detect best_model.pt)
    if checkpoint_path.is_dir():
        for name in ['best_model.pt', 'latest_model.pt']:
            if (checkpoint_path / name).exists():
                checkpoint_path = checkpoint_path / name
                break
    
    print(f"Loading checkpoint: {checkpoint_path}")
    
    checkpoint = torch.load(str(checkpoint_path), map_location=device, weights_only=False)
    config = RankingConfig(**checkpoint['config'])
    
    model = SiameseRanker(config)
    model.load_state_dict(checkpoint['model_state_dict'])
    model = model.to(device)
    model.eval()
    
    # Count parameters
    num_params = sum(p.numel() for p in model.parameters())
    
    print(f"  Loaded from epoch {checkpoint['epoch'] + 1}")
    print(f"  Best val AUC: {checkpoint.get('val_auc', 'N/A')}")
    print(f"  Model parameters: {num_params:,}")
    
    return model, config, checkpoint

# Load model
model, config, checkpoint = load_model_and_config(CHECKPOINT_PATH, DEVICE)
history = checkpoint.get('history', {})

In [ ]:
# Load scenario normalization stats
stats_path = Path(CHECKPOINT_PATH)
if stats_path.is_file():
    stats_path = stats_path.parent / "scenario_stats.json"
else:
    stats_path = stats_path / "scenario_stats.json"

if stats_path.exists():
    with open(stats_path, 'r') as f:
        stats_data = json.load(f)
        if 'scenario_stats' in stats_data:
            scenario_stats = stats_data['scenario_stats']
        else:
            scenario_stats = stats_data
    print(f"Loaded scenario stats from: {stats_path}")
    print(f"  Means: {scenario_stats['means']}")
    print(f"  Stds: {scenario_stats['stds']}")
else:
    # Compute from training data
    print("Computing scenario stats from training data...")
    scenario_stats = compute_scenario_stats(str(DATA_DIR / "train_pairs.jsonl"))
    print(f"  Means: {scenario_stats['means']}")
    print(f"  Stds: {scenario_stats['stds']}")

In [ ]:
# Create test dataloader for pairwise evaluation
test_pairs_file = DATA_DIR / "test_pairs.jsonl"

test_dataset = PairwiseDataset(
    pairs_file=str(test_pairs_file),
    floor_plans_dir=str(FLOOR_PLANS_DIR),
    target_size=config.target_grid_size,
    scenario_stats=scenario_stats,
    augment=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=EVAL_BATCH_SIZE,
    shuffle=False,
    num_workers=0,  # Use 0 for notebook compatibility
    pin_memory=True if DEVICE.type == 'cuda' else False
)

print(f"Test pairs loaded: {len(test_dataset):,}")

In [ ]:
# Create SingleConfigDataset for per-plan evaluation
simulation_results_file = DATA_DIR / "simulation_results.jsonl"

eval_dataset = SingleConfigDataset(
    simulation_results_file=str(simulation_results_file),
    floor_plans_dir=str(FLOOR_PLANS_DIR),
    target_size=config.target_grid_size,
    scenario_stats=scenario_stats,
    max_configs=None  # Load all
)

print(f"Evaluation configs loaded: {len(eval_dataset):,}")

---
## 2. Pairwise Evaluation

Evaluate the model on test pairs measuring:
- **Loss**: RankNet BCE loss on test set
- **Accuracy**: Percentage of correctly ordered pairs
- **AUC**: Area under ROC curve for pair classification
- **Weighted Accuracy**: Accuracy weighted by label confidence

In [ ]:
@torch.no_grad()
def evaluate_pairwise_detailed(
    model: SiameseRanker,
    test_loader: DataLoader,
    config: RankingConfig,
    device: torch.device
) -> Dict:
    """Detailed pairwise evaluation with additional metrics."""
    model.eval()
    
    criterion = RankNetLoss(sigma=config.sigma)
    
    all_logits = []
    all_labels = []
    all_confidences = []
    all_score_diffs = []  # Ground truth score differences
    total_loss = 0.0
    num_batches = 0
    
    for batch in test_loader:
        grid_a = batch['grid_a'].to(device)
        scenario_a = batch['scenario_a'].to(device)
        grid_b = batch['grid_b'].to(device)
        scenario_b = batch['scenario_b'].to(device)
        label = batch['label'].to(device)
        confidence = batch['confidence'].to(device)
        score_a_gt = batch['score_a']
        score_b_gt = batch['score_b']
        
        # Forward pass
        _, _, logit = model(grid_a, scenario_a, grid_b, scenario_b)
        
        # Compute loss
        loss = criterion(logit, label, confidence)
        total_loss += loss.item()
        num_batches += 1
        
        all_logits.extend(logit.cpu().numpy())
        all_labels.extend(label.cpu().numpy())
        all_confidences.extend(confidence.cpu().numpy())
        all_score_diffs.extend((score_a_gt - score_b_gt).numpy())
    
    # Convert to numpy
    all_logits = np.array(all_logits)
    all_labels = np.array(all_labels)
    all_confidences = np.array(all_confidences)
    all_score_diffs = np.array(all_score_diffs)
    
    # Compute metrics
    predictions = (all_logits > 0).astype(int)
    accuracy = (predictions == all_labels).mean()
    
    # AUC
    try:
        auc = roc_auc_score(all_labels, all_logits)
    except ValueError:
        auc = 0.5
    
    # Weighted accuracy
    weighted_correct = ((predictions == all_labels) * all_confidences).sum()
    weighted_accuracy = weighted_correct / all_confidences.sum()
    
    # ROC curve data
    fpr, tpr, thresholds = roc_curve(all_labels, all_logits)
    
    return {
        'loss': total_loss / num_batches,
        'accuracy': accuracy,
        'auc': auc,
        'weighted_accuracy': weighted_accuracy,
        'num_pairs': len(all_labels),
        'logits': all_logits,
        'labels': all_labels,
        'confidences': all_confidences,
        'score_diffs_gt': all_score_diffs,
        'fpr': fpr,
        'tpr': tpr,
        'thresholds': thresholds
    }

In [ ]:
# Run pairwise evaluation
print("Evaluating on test set...")
pairwise_results = evaluate_pairwise_detailed(model, test_loader, config, DEVICE)

print(f"\n{'='*50}")
print("PAIRWISE METRICS")
print(f"{'='*50}")
print(f"Loss:              {pairwise_results['loss']:.4f}")
print(f"Accuracy:          {pairwise_results['accuracy']:.4f}")
print(f"AUC:               {pairwise_results['auc']:.4f}")
print(f"Weighted Accuracy: {pairwise_results['weighted_accuracy']:.4f}")
print(f"Total Pairs:       {pairwise_results['num_pairs']:,}")

In [ ]:
# Visualize pairwise results
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# 1. Logit Distribution
ax = axes[0, 0]
logits_pos = pairwise_results['logits'][pairwise_results['labels'] == 1]
logits_neg = pairwise_results['logits'][pairwise_results['labels'] == 0]
ax.hist(logits_neg, bins=50, alpha=0.6, label='Label=0 (B>A)', color='red', density=True)
ax.hist(logits_pos, bins=50, alpha=0.6, label='Label=1 (A>B)', color='green', density=True)
ax.axvline(x=0, color='black', linestyle='--', label='Decision boundary')
ax.set_xlabel('Logit (s(A) - s(B))')
ax.set_ylabel('Density')
ax.set_title('Logit Distribution by True Label')
ax.legend()

# 2. ROC Curve
ax = axes[0, 1]
ax.plot(pairwise_results['fpr'], pairwise_results['tpr'], 'b-', linewidth=2, 
        label=f'ROC (AUC = {pairwise_results["auc"]:.4f})')
ax.plot([0, 1], [0, 1], 'r--', label='Random baseline')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve')
ax.legend()
ax.grid(True, alpha=0.3)

# 3. Accuracy by Confidence Bin
ax = axes[1, 0]
conf_bins = np.linspace(0, 1, 11)
conf_labels = [f'{conf_bins[i]:.1f}-{conf_bins[i+1]:.1f}' for i in range(len(conf_bins)-1)]
bin_accs = []
bin_counts = []
for i in range(len(conf_bins) - 1):
    mask = (pairwise_results['confidences'] >= conf_bins[i]) & \
           (pairwise_results['confidences'] < conf_bins[i+1])
    if mask.sum() > 0:
        bin_acc = ((pairwise_results['logits'][mask] > 0).astype(int) == 
                   pairwise_results['labels'][mask]).mean()
        bin_accs.append(bin_acc)
        bin_counts.append(mask.sum())
    else:
        bin_accs.append(0)
        bin_counts.append(0)
        
ax.bar(range(len(bin_accs)), bin_accs, color='steelblue', alpha=0.8)
ax.set_xticks(range(len(conf_labels)))
ax.set_xticklabels(conf_labels, rotation=45, ha='right')
ax.set_xlabel('Label Confidence Bin')
ax.set_ylabel('Accuracy')
ax.set_title('Accuracy by Label Confidence')
ax.set_ylim(0, 1)
ax.axhline(y=0.5, color='red', linestyle='--', alpha=0.5)

# 4. Predicted vs Ground Truth Score Difference Correlation
ax = axes[1, 1]
ax.scatter(pairwise_results['score_diffs_gt'], pairwise_results['logits'], 
           alpha=0.1, s=5, color='steelblue')
correlation = np.corrcoef(pairwise_results['score_diffs_gt'], pairwise_results['logits'])[0, 1]
ax.set_xlabel('Ground Truth Score Diff (score_a - score_b)')
ax.set_ylabel('Predicted Logit (s(A) - s(B))')
ax.set_title(f'Logit vs GT Score Diff (r = {correlation:.3f})')
ax.axhline(y=0, color='black', linestyle='--', alpha=0.5)
ax.axvline(x=0, color='black', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig('pairwise_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 3. Surrogate vs Simulation Comparison

Compare model predicted scores against actual fire simulation results:
- Scatter plot: Predicted score vs Ground truth composite score
- Correlation metrics: R², Pearson, Spearman
- Per-metric analysis: survival_rate, steps, fire_damage

In [ ]:
@torch.no_grad()
def collect_predictions_and_metrics(
    model: SiameseRanker,
    dataset: SingleConfigDataset,
    device: torch.device,
    batch_size: int = 256
) -> pd.DataFrame:
    """Collect model predictions and ground truth metrics."""
    model.eval()
    
    all_data = []
    
    # Batch processing
    for i in range(0, len(dataset), batch_size):
        batch_indices = range(i, min(i + batch_size, len(dataset)))
        
        grids = torch.stack([dataset[j]['grid'] for j in batch_indices]).to(device)
        scenarios = torch.stack([dataset[j]['scenario'] for j in batch_indices]).to(device)
        
        # Get predictions
        predicted_scores = model.score_single(grids, scenarios).cpu().numpy()
        
        # Collect data
        for idx, j in enumerate(batch_indices):
            item = dataset[j]
            all_data.append({
                'index': j,
                'floor_plan_id': item['floor_plan_id'],
                'predicted_score': predicted_scores[idx],
                'ground_truth_score': item['ground_truth_score'],
                'survival_rate': item['survival_rate'],
                'steps': item['steps'],
                'avg_fire_damage': item['avg_fire_damage']
            })
        
        if (i + batch_size) % 5000 < batch_size:
            print(f"Processed {min(i + batch_size, len(dataset)):,} / {len(dataset):,} configs")
    
    return pd.DataFrame(all_data)

In [ ]:
# Collect predictions
print("Collecting predictions and ground truth metrics...")
results_df = collect_predictions_and_metrics(model, eval_dataset, DEVICE)
print(f"Collected {len(results_df):,} samples")
results_df.head()

In [ ]:
def compute_correlations(df: pd.DataFrame) -> Dict:
    """Compute correlation metrics between predicted and ground truth."""
    pred = df['predicted_score'].values
    gt = df['ground_truth_score'].values
    
    # Correlation with composite score
    pearson_r, pearson_p = pearsonr(pred, gt)
    spearman_r, spearman_p = spearmanr(pred, gt)
    r_squared = pearson_r ** 2
    
    # Correlation with individual metrics
    survival_r, _ = pearsonr(pred, df['survival_rate'].values)
    steps_r, _ = pearsonr(pred, df['steps'].values)
    fire_damage_r, _ = pearsonr(pred, df['avg_fire_damage'].values)
    
    return {
        'r_squared': r_squared,
        'pearson_r': pearson_r,
        'pearson_p': pearson_p,
        'spearman_r': spearman_r,
        'spearman_p': spearman_p,
        'survival_rate_r': survival_r,
        'steps_r': steps_r,
        'fire_damage_r': fire_damage_r
    }

correlations = compute_correlations(results_df)

print(f"\n{'='*50}")
print("CORRELATION METRICS")
print(f"{'='*50}")
print(f"R-squared:            {correlations['r_squared']:.4f}")
print(f"Pearson r:            {correlations['pearson_r']:.4f} (p={correlations['pearson_p']:.2e})")
print(f"Spearman rho:         {correlations['spearman_r']:.4f} (p={correlations['spearman_p']:.2e})")
print(f"\nPer-Metric Correlations:")
print(f"  Survival Rate:      {correlations['survival_rate_r']:.4f}")
print(f"  Steps (neg better): {correlations['steps_r']:.4f}")
print(f"  Fire Damage (neg):  {correlations['fire_damage_r']:.4f}")

In [ ]:
# Visualize surrogate vs simulation
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# 1. Predicted vs Ground Truth Score (scatter)
ax = axes[0, 0]
ax.scatter(results_df['ground_truth_score'], results_df['predicted_score'], 
           alpha=0.1, s=5, color='steelblue')
# Add regression line
z = np.polyfit(results_df['ground_truth_score'], results_df['predicted_score'], 1)
p = np.poly1d(z)
x_line = np.linspace(results_df['ground_truth_score'].min(), results_df['ground_truth_score'].max(), 100)
ax.plot(x_line, p(x_line), 'r-', linewidth=2, label=f'Linear fit (R²={correlations["r_squared"]:.3f})')
ax.set_xlabel('Ground Truth Composite Score')
ax.set_ylabel('Predicted Score')
ax.set_title('Predicted vs Ground Truth Score')
ax.legend()
ax.grid(True, alpha=0.3)

# 2. Predicted Score vs Survival Rate
ax = axes[0, 1]
ax.scatter(results_df['survival_rate'], results_df['predicted_score'], 
           alpha=0.1, s=5, color='green')
ax.set_xlabel('Survival Rate')
ax.set_ylabel('Predicted Score')
ax.set_title(f'Predicted Score vs Survival Rate (r={correlations["survival_rate_r"]:.3f})')
ax.grid(True, alpha=0.3)

# 3. Predicted Score vs Steps
ax = axes[1, 0]
ax.scatter(results_df['steps'], results_df['predicted_score'], 
           alpha=0.1, s=5, color='orange')
ax.set_xlabel('Evacuation Steps')
ax.set_ylabel('Predicted Score')
ax.set_title(f'Predicted Score vs Steps (r={correlations["steps_r"]:.3f})')
ax.grid(True, alpha=0.3)

# 4. Predicted Score vs Fire Damage
ax = axes[1, 1]
ax.scatter(results_df['avg_fire_damage'], results_df['predicted_score'], 
           alpha=0.1, s=5, color='red')
ax.set_xlabel('Average Fire Damage')
ax.set_ylabel('Predicted Score')
ax.set_title(f'Predicted Score vs Fire Damage (r={correlations["fire_damage_r"]:.3f})')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('surrogate_vs_simulation.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Error Analysis
# Normalize predicted scores to same scale as ground truth for comparison
pred_min, pred_max = results_df['predicted_score'].min(), results_df['predicted_score'].max()
gt_min, gt_max = results_df['ground_truth_score'].min(), results_df['ground_truth_score'].max()
results_df['predicted_normalized'] = (results_df['predicted_score'] - pred_min) / (pred_max - pred_min)
results_df['predicted_normalized'] = results_df['predicted_normalized'] * (gt_max - gt_min) + gt_min
results_df['normalized_error'] = results_df['predicted_normalized'] - results_df['ground_truth_score']
results_df['abs_error'] = results_df['normalized_error'].abs()

print(f"\n{'='*50}")
print("ERROR STATISTICS (Normalized)")
print(f"{'='*50}")
print(f"Mean Absolute Error:  {results_df['abs_error'].mean():.4f}")
print(f"RMSE:                 {np.sqrt((results_df['normalized_error']**2).mean()):.4f}")
print(f"Error Std:            {results_df['normalized_error'].std():.4f}")
print(f"Error Range:          [{results_df['normalized_error'].min():.4f}, {results_df['normalized_error'].max():.4f}]")

In [ ]:
# Find worst predictions
worst_indices = results_df['abs_error'].nlargest(10).index
print("\nTop 10 Worst Predictions:")
display(results_df.loc[worst_indices, ['index', 'floor_plan_id', 'predicted_normalized', 
                                        'ground_truth_score', 'normalized_error', 
                                        'survival_rate', 'steps', 'avg_fire_damage']])

---
## 4. Per-Plan Ranking Evaluation

Evaluate ranking quality within each floor plan:
- **Kendall Tau**: Rank correlation coefficient
- **Spearman rho**: Rank correlation
- **NDCG@5**: Normalized Discounted Cumulative Gain at top-5
- **Top-1 Accuracy**: Is the predicted best also the true best?

In [ ]:
def evaluate_per_plan_ranking_detailed(df: pd.DataFrame, k: int = 5) -> Tuple[Dict, pd.DataFrame]:
    """Compute per-plan ranking metrics."""
    
    plan_metrics = []
    
    for plan_id, group in df.groupby('floor_plan_id'):
        if len(group) < 2:
            continue
            
        pred_scores = group['predicted_score'].values
        true_scores = group['ground_truth_score'].values
        
        # Kendall Tau
        tau, tau_p = kendalltau(pred_scores, true_scores)
        
        # Spearman
        rho, rho_p = spearmanr(pred_scores, true_scores)
        
        # NDCG@k
        ndcg = compute_ndcg(pred_scores, true_scores, k=min(k, len(pred_scores)))
        
        # Top-1 accuracy
        top1_correct = int(np.argmax(pred_scores) == np.argmax(true_scores))
        
        plan_metrics.append({
            'floor_plan_id': plan_id,
            'num_configs': len(group),
            'kendall_tau': tau if not np.isnan(tau) else 0.0,
            'spearman_rho': rho if not np.isnan(rho) else 0.0,
            'ndcg': ndcg,
            'top1_correct': top1_correct
        })
    
    metrics_df = pd.DataFrame(plan_metrics)
    
    # Aggregate metrics
    aggregated = {
        'mean_kendall_tau': metrics_df['kendall_tau'].mean(),
        'std_kendall_tau': metrics_df['kendall_tau'].std(),
        'mean_spearman_rho': metrics_df['spearman_rho'].mean(),
        'std_spearman_rho': metrics_df['spearman_rho'].std(),
        'mean_ndcg': metrics_df['ndcg'].mean(),
        'std_ndcg': metrics_df['ndcg'].std(),
        'top1_accuracy': metrics_df['top1_correct'].mean(),
        'num_plans': len(metrics_df)
    }
    
    return aggregated, metrics_df

In [ ]:
# Compute per-plan metrics
ranking_metrics, per_plan_df = evaluate_per_plan_ranking_detailed(results_df, k=5)

print(f"\n{'='*50}")
print("PER-PLAN RANKING METRICS")
print(f"{'='*50}")
print(f"Kendall Tau:    {ranking_metrics['mean_kendall_tau']:.4f} (+/- {ranking_metrics['std_kendall_tau']:.4f})")
print(f"Spearman rho:   {ranking_metrics['mean_spearman_rho']:.4f} (+/- {ranking_metrics['std_spearman_rho']:.4f})")
print(f"NDCG@5:         {ranking_metrics['mean_ndcg']:.4f} (+/- {ranking_metrics['std_ndcg']:.4f})")
print(f"Top-1 Accuracy: {ranking_metrics['top1_accuracy']:.4f}")
print(f"Floor Plans:    {ranking_metrics['num_plans']:,}")

In [ ]:
# Visualize per-plan metrics
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Kendall Tau Distribution
ax = axes[0, 0]
ax.hist(per_plan_df['kendall_tau'], bins=30, color='steelblue', edgecolor='white', alpha=0.8)
ax.axvline(x=per_plan_df['kendall_tau'].mean(), color='red', linestyle='--', 
           label=f'Mean = {per_plan_df["kendall_tau"].mean():.3f}')
ax.set_xlabel('Kendall Tau')
ax.set_ylabel('Count')
ax.set_title('Distribution of Kendall Tau per Floor Plan')
ax.legend()

# 2. Spearman Rho Distribution
ax = axes[0, 1]
ax.hist(per_plan_df['spearman_rho'], bins=30, color='green', edgecolor='white', alpha=0.8)
ax.axvline(x=per_plan_df['spearman_rho'].mean(), color='red', linestyle='--',
           label=f'Mean = {per_plan_df["spearman_rho"].mean():.3f}')
ax.set_xlabel('Spearman Rho')
ax.set_ylabel('Count')
ax.set_title('Distribution of Spearman Rho per Floor Plan')
ax.legend()

# 3. NDCG Distribution
ax = axes[1, 0]
ax.hist(per_plan_df['ndcg'], bins=30, color='orange', edgecolor='white', alpha=0.8)
ax.axvline(x=per_plan_df['ndcg'].mean(), color='red', linestyle='--',
           label=f'Mean = {per_plan_df["ndcg"].mean():.3f}')
ax.set_xlabel('NDCG@5')
ax.set_ylabel('Count')
ax.set_title('Distribution of NDCG@5 per Floor Plan')
ax.legend()

# 4. Metrics vs Number of Configs
ax = axes[1, 1]
ax.scatter(per_plan_df['num_configs'], per_plan_df['kendall_tau'], alpha=0.5, s=20, label='Kendall Tau')
ax.scatter(per_plan_df['num_configs'], per_plan_df['ndcg'], alpha=0.5, s=20, label='NDCG@5')
ax.set_xlabel('Number of Configs per Plan')
ax.set_ylabel('Metric Value')
ax.set_title('Ranking Quality vs Number of Configs')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('per_plan_ranking.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 5. GradCAM Visualization

Visualize model attention using Gradient-weighted Class Activation Mapping:

**Selection Modes:**
- Test set samples (by index)
- Best predictions (lowest error)
- Worst predictions (highest error)
- Random sampling

In [ ]:
def select_samples_for_gradcam(
    df: pd.DataFrame,
    mode: str = 'random',
    n_samples: int = 6,
    seed: int = 42
) -> List[int]:
    """
    Select sample indices for GradCAM visualization.
    
    Args:
        df: Results dataframe with errors and indices
        mode: Selection mode - 'random', 'best', 'worst', 'range'
        n_samples: Number of samples to select
        seed: Random seed
        
    Returns:
        List of dataset indices
    """
    np.random.seed(seed)
    
    if mode == 'random':
        indices = np.random.choice(len(df), size=min(n_samples, len(df)), replace=False)
        return df.iloc[sorted(indices)]['index'].tolist()
    
    elif mode == 'best':
        # Lowest absolute error
        best_rows = df.nsmallest(n_samples, 'abs_error')
        return best_rows['index'].tolist()
    
    elif mode == 'worst':
        # Highest absolute error
        worst_rows = df.nlargest(n_samples, 'abs_error')
        return worst_rows['index'].tolist()
    
    elif mode == 'range':
        # Evenly spaced across the dataset
        step = len(df) // n_samples
        row_indices = [i * step for i in range(n_samples)]
        return df.iloc[row_indices]['index'].tolist()
    
    else:
        raise ValueError(f"Unknown mode: {mode}")

In [ ]:
def visualize_gradcam_grid(
    model: SiameseRanker,
    dataset: SingleConfigDataset,
    indices: List[int],
    device: torch.device,
    cols: int = 3,
    figsize: Tuple[int, int] = None
):
    """Generate a grid of GradCAM visualizations."""
    n_samples = len(indices)
    rows = (n_samples + cols - 1) // cols
    
    if figsize is None:
        figsize = (5 * cols, 5 * rows)
    
    fig, axes = plt.subplots(rows, cols, figsize=figsize)
    if rows == 1:
        axes = axes.reshape(1, -1)
    if cols == 1:
        axes = axes.reshape(-1, 1)
    
    gradcam = GradCAM(model)
    
    for idx, sample_idx in enumerate(indices):
        row, col = idx // cols, idx % cols
        
        sample = dataset[sample_idx]
        grid = sample['grid'].to(device)
        scenario = sample['scenario'].to(device)
        
        # Generate GradCAM
        cam = gradcam.generate(grid, scenario)
        
        # Get predicted score
        with torch.no_grad():
            pred_score = model.score_single(
                grid.unsqueeze(0), scenario.unsqueeze(0)
            ).item()
        
        # Floor plan (passable channel)
        floor_plan = grid[1].cpu().numpy()
        
        # Plot overlay
        ax = axes[row, col]
        ax.imshow(floor_plan, cmap='gray')
        ax.imshow(cam, cmap='jet', alpha=0.5)
        ax.set_title(f'Idx {sample_idx}\nPred: {pred_score:.3f} | GT: {sample["ground_truth_score"]:.3f}', 
                     fontsize=10)
        ax.axis('off')
    
    # Hide unused axes
    for idx in range(n_samples, rows * cols):
        row, col = idx // cols, idx % cols
        axes[row, col].axis('off')
    
    plt.tight_layout()
    return fig

In [ ]:
# GradCAM - Random Samples
print("GradCAM Visualization: Random Samples")
random_indices = select_samples_for_gradcam(results_df, mode='random', n_samples=6)
fig = visualize_gradcam_grid(model, eval_dataset, random_indices, DEVICE, cols=3)
plt.savefig('gradcam_random.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Selected indices: {random_indices}")

In [ ]:
# GradCAM - Best Predictions (lowest error)
print("GradCAM Visualization: Best Predictions (Lowest Error)")
best_indices = select_samples_for_gradcam(results_df, mode='best', n_samples=6)
fig = visualize_gradcam_grid(model, eval_dataset, best_indices, DEVICE, cols=3)
plt.savefig('gradcam_best.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Selected indices: {best_indices}")

In [ ]:
# GradCAM - Worst Predictions (highest error)
print("GradCAM Visualization: Worst Predictions (Highest Error)")
worst_indices = select_samples_for_gradcam(results_df, mode='worst', n_samples=6)
fig = visualize_gradcam_grid(model, eval_dataset, worst_indices, DEVICE, cols=3)
plt.savefig('gradcam_worst.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Selected indices: {worst_indices}")

In [ ]:
# Interactive: Visualize a single sample with detailed GradCAM
def visualize_single_sample(sample_idx: int):
    """Visualize a single sample with detailed GradCAM."""
    sample = eval_dataset[sample_idx]
    grid = sample['grid'].to(DEVICE)
    scenario = sample['scenario'].to(DEVICE)
    
    gradcam = GradCAM(model)
    cam = gradcam.generate(grid, scenario)
    
    with torch.no_grad():
        pred_score = model.score_single(grid.unsqueeze(0), scenario.unsqueeze(0)).item()
    
    # Floor plan channels
    fig, axes = plt.subplots(1, 5, figsize=(20, 4))
    channel_names = ['Wall', 'Passable', 'Doors', 'Exits', 'Valid Mask']
    for i, name in enumerate(channel_names):
        axes[i].imshow(grid[i].cpu().numpy(), cmap='gray')
        axes[i].set_title(name)
        axes[i].axis('off')
    
    plt.suptitle(f'Sample {sample_idx} | Predicted: {pred_score:.4f} | GT: {sample["ground_truth_score"]:.4f}')
    plt.tight_layout()
    plt.show()
    
    # GradCAM overlay
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    floor_plan = grid[1].cpu().numpy()
    
    axes[0].imshow(floor_plan, cmap='gray')
    axes[0].set_title('Floor Plan')
    axes[0].axis('off')
    
    axes[1].imshow(cam, cmap='jet')
    axes[1].set_title('GradCAM Attention')
    axes[1].axis('off')
    
    axes[2].imshow(floor_plan, cmap='gray')
    axes[2].imshow(cam, cmap='jet', alpha=0.5)
    axes[2].set_title('Overlay')
    axes[2].axis('off')
    
    plt.tight_layout()
    plt.show()
    
    print(f"Sample Details:")
    print(f"  Floor Plan ID: {sample['floor_plan_id']}")
    print(f"  Survival Rate: {sample['survival_rate']:.4f}")
    print(f"  Steps: {sample['steps']}")
    print(f"  Fire Damage: {sample['avg_fire_damage']:.4f}")

In [ ]:
# Example usage - change index as needed
# Uncomment and modify the index to visualize a specific sample
# visualize_single_sample(0)

---
## 6. Summary & Export

Aggregate all metrics and export results.

In [ ]:
# Create summary table
summary_data = {
    'Metric': [
        # Pairwise
        'Test Loss (RankNet)',
        'Pairwise Accuracy',
        'AUC',
        'Weighted Accuracy',
        # Correlation
        'R-squared (Pred vs GT)',
        'Pearson r',
        'Spearman rho',
        # Per-metric correlation
        'Survival Rate Correlation',
        'Steps Correlation',
        'Fire Damage Correlation',
        # Per-plan ranking
        'Mean Kendall Tau',
        'Mean Spearman Rho',
        'Mean NDCG@5',
        'Top-1 Accuracy',
        # Dataset info
        'Test Pairs',
        'Total Configs',
        'Floor Plans'
    ],
    'Value': [
        f"{pairwise_results['loss']:.4f}",
        f"{pairwise_results['accuracy']:.4f}",
        f"{pairwise_results['auc']:.4f}",
        f"{pairwise_results['weighted_accuracy']:.4f}",
        f"{correlations['r_squared']:.4f}",
        f"{correlations['pearson_r']:.4f}",
        f"{correlations['spearman_r']:.4f}",
        f"{correlations['survival_rate_r']:.4f}",
        f"{correlations['steps_r']:.4f}",
        f"{correlations['fire_damage_r']:.4f}",
        f"{ranking_metrics['mean_kendall_tau']:.4f} (+/- {ranking_metrics['std_kendall_tau']:.4f})",
        f"{ranking_metrics['mean_spearman_rho']:.4f} (+/- {ranking_metrics['std_spearman_rho']:.4f})",
        f"{ranking_metrics['mean_ndcg']:.4f} (+/- {ranking_metrics['std_ndcg']:.4f})",
        f"{ranking_metrics['top1_accuracy']:.4f}",
        f"{pairwise_results['num_pairs']:,}",
        f"{len(results_df):,}",
        f"{ranking_metrics['num_plans']:,}"
    ]
}

summary_df = pd.DataFrame(summary_data)
display(summary_df.style.hide(axis='index'))

In [ ]:
# Export results to JSON
export_data = {
    'checkpoint': str(CHECKPOINT_PATH),
    'epoch': checkpoint['epoch'] + 1,
    'pairwise': {
        'loss': float(pairwise_results['loss']),
        'accuracy': float(pairwise_results['accuracy']),
        'auc': float(pairwise_results['auc']),
        'weighted_accuracy': float(pairwise_results['weighted_accuracy']),
        'num_pairs': int(pairwise_results['num_pairs'])
    },
    'correlation': {
        'r_squared': float(correlations['r_squared']),
        'pearson_r': float(correlations['pearson_r']),
        'spearman_r': float(correlations['spearman_r']),
        'survival_rate_r': float(correlations['survival_rate_r']),
        'steps_r': float(correlations['steps_r']),
        'fire_damage_r': float(correlations['fire_damage_r'])
    },
    'per_plan_ranking': {
        'mean_kendall_tau': float(ranking_metrics['mean_kendall_tau']),
        'std_kendall_tau': float(ranking_metrics['std_kendall_tau']),
        'mean_spearman_rho': float(ranking_metrics['mean_spearman_rho']),
        'std_spearman_rho': float(ranking_metrics['std_spearman_rho']),
        'mean_ndcg': float(ranking_metrics['mean_ndcg']),
        'std_ndcg': float(ranking_metrics['std_ndcg']),
        'top1_accuracy': float(ranking_metrics['top1_accuracy']),
        'num_plans': int(ranking_metrics['num_plans'])
    },
    'error_stats': {
        'mae': float(results_df['abs_error'].mean()),
        'rmse': float(np.sqrt((results_df['normalized_error']**2).mean())),
        'std': float(results_df['normalized_error'].std())
    }
}

# Save to JSON
output_path = 'evaluation_results.json'
with open(output_path, 'w') as f:
    json.dump(export_data, f, indent=2)
print(f"Results saved to {output_path}")

# Save detailed results to CSV
results_df.to_csv('detailed_predictions.csv', index=False)
per_plan_df.to_csv('per_plan_metrics.csv', index=False)
print("Detailed results saved to CSV files")

In [ ]:
# Plot training history if available
if history and 'train_loss' in history:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    epochs = range(1, len(history['train_loss']) + 1)
    
    # Loss
    ax = axes[0, 0]
    ax.plot(epochs, history['train_loss'], label='Train Loss')
    ax.plot(epochs, history['val_loss'], label='Val Loss')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.set_title('Training and Validation Loss')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # Accuracy
    ax = axes[0, 1]
    ax.plot(epochs, history['train_accuracy'], label='Train Acc')
    ax.plot(epochs, history['val_accuracy'], label='Val Acc')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Accuracy')
    ax.set_title('Training and Validation Accuracy')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # AUC
    ax = axes[1, 0]
    ax.plot(epochs, history['val_auc'], label='Val AUC', color='green')
    ax.axhline(y=0.5, color='red', linestyle='--', label='Random')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('AUC')
    ax.set_title('Validation AUC')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # Learning Rate
    ax = axes[1, 1]
    if 'learning_rate' in history:
        ax.plot(epochs, history['learning_rate'], color='orange')
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Learning Rate')
        ax.set_title('Learning Rate Schedule')
        ax.set_yscale('log')
        ax.grid(True, alpha=0.3)
    else:
        ax.text(0.5, 0.5, 'Learning rate history not available', 
                ha='center', va='center', transform=ax.transAxes)
        ax.axis('off')
    
    plt.tight_layout()
    plt.savefig('training_history.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("No training history available in checkpoint")

---
## Done!

All evaluation results have been computed and exported. Check the generated files:
- `evaluation_results.json` - Aggregated metrics
- `detailed_predictions.csv` - Per-config predictions
- `per_plan_metrics.csv` - Per-plan ranking metrics
- `*.png` - Visualization images